# Python 基础：面向 AI Infra 的工程入门

本 Notebook 根据 [AIInfraGuide：第1章 编程语言基础](https://caomaolufei.github.io/AIInfraGuide/guides/%E6%A8%A1%E5%9D%97%E4%B8%80-%E5%89%8D%E7%BD%AE%E7%9F%A5%E8%AF%86/%E7%AC%AC1%E7%AB%A0-%E7%BC%96%E7%A8%8B%E8%AF%AD%E8%A8%80%E5%9F%BA%E7%A1%80/) 的 Python 部分整理并改编。

学习目标：理解 Python 对象模型、常用工程特性、并发模型与性能分析方法。建议从上到下依次运行。

In [5]:
import platform
import sys

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("平台:", platform.platform())

Python: 3.9.23
解释器: e:\Users\ASUS\anaconda3\envs\transformers\python.exe
平台: Windows-10-10.0.26100-SP0


## 1. 名字、对象与引用

Python 的变量名绑定到对象。把一个列表赋给另一个名字，通常不会复制列表；`is` 检查是否为同一个对象，`==` 检查值是否相等。

In [6]:
a = [1, 2]
b = a
b.append(3)

print("a =", a)
print("a is b:", a is b)
print("a == b:", a == b)

a = [1, 2, 3]
a is b: True
a == b: True


In [7]:
def mutate(values: list[int]) -> None:
    values.append(1)

def rebind(values: list[int]) -> None:
    values = [99]  # 只改变局部名字的绑定

data: list[int] = []
mutate(data)
print("mutate 后:", data)
rebind(data)
print("rebind 后:", data)

mutate 后: [1]
rebind 后: [1]


## 2. 可变性、默认参数与拷贝

列表、字典和集合通常可变；数字、字符串等通常不可变。可变默认参数在函数定义时只创建一次，因此容易意外共享状态。

In [ ]:
# 错误：同一个列表会被多次调用共享
def collect_bad(x: int, result: list[int] = []) -> list[int]:
    result.append(x)
    return result

# 正确：用 None 表示“未提供”
def collect(x: int, result: list[int] | None = None) -> list[int]:
    if result is None:
        result = []
    result.append(x)
    return result


In [14]:
import copy

source = [[1], [2]]
shallow = source.copy()
deep = copy.deepcopy(source)
shallow[0].append(9)

print("原对象:", source)
print("浅拷贝:", shallow)
print("深拷贝:", deep)

原对象: [[1, 9], [2]]
浅拷贝: [[1, 9], [2]]
深拷贝: [[1], [2]]


> AI Infra 提示：大型张量和模型状态的复制成本很高。复制之前，应先确定数据是否真的需要独占。

## 3. 作用域与闭包

名字按 LEGB 顺序查找：局部、外层函数、模块全局、内置作用域。闭包会保留对外层环境的引用。

In [15]:
from collections.abc import Callable

def make_multiplier(scale: float) -> Callable[[float], float]:
    def multiply(value: float) -> float:
        return value * scale
    return multiply

double = make_multiplier(2.0)
print(double(3.0))

late_binding = [lambda: i for i in range(3)]
captured = [lambda i=i: i for i in range(3)]
print("延迟绑定:", [fn() for fn in late_binding])
print("立即捕获:", [fn() for fn in captured])

6.0
延迟绑定: [2, 2, 2]
立即捕获: [0, 1, 2]


## 4. 类、数据类与协议

特殊方法让自定义对象接入 Python 协议。例如，`__len__` 支持 `len()`，`__iter__` 支持 `for` 循环。

In [ ]:
from collections.abc import Iterator
from dataclasses import dataclass

@dataclass
class Batch:
    samples: list[list[float]]

    def __len__(self) -> int:
        return len(self.samples)

    def __iter__(self) -> Iterator[list[float]]:
        return iter(self.samples)

batch = Batch([[1.0, 2.0], [3.0, 4.0]])
print("批大小:", len(batch))
for sample in batch:
    print("样本:", sample)

In [ ]:
@dataclass
class Device:
    kind: str
    index: int

    @classmethod
    def parse(cls, text: str) -> "Device":
        kind, index = text.split(":")
        return cls(kind, int(index))

device = Device.parse("cuda:0")
print(device)

## 5. 迭代器与生成器

生成器用 `yield` 按需产出数据，适合无法一次全部装入内存的数据流。生成器通常只能完整消费一次。

In [ ]:
from collections.abc import Iterable, Iterator

def make_batches(items: Iterable[str], batch_size: int) -> Iterator[list[str]]:
    if batch_size <= 0:
        raise ValueError("batch_size 必须大于 0")
    batch: list[str] = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

for current_batch in make_batches((str(i) for i in range(7)), 3):
    print(current_batch)

## 6. 装饰器与上下文管理器

装饰器适合在函数调用边界添加日志、缓存或计时；上下文管理器用 `with` 明确资源的获取和释放范围。

In [ ]:
from collections.abc import Callable
from functools import wraps
from time import perf_counter
from typing import ParamSpec, TypeVar

P = ParamSpec("P")
R = TypeVar("R")

def timed(function: Callable[P, R]) -> Callable[P, R]:
    @wraps(function)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        start = perf_counter()
        try:
            return function(*args, **kwargs)
        finally:
            elapsed_ms = (perf_counter() - start) * 1_000
            print(f"{function.__name__}: {elapsed_ms:.3f} ms")
    return wrapper

@timed
def preprocess(values: list[float]) -> list[float]:
    return [value * 2 for value in values]

print(preprocess([1.0, 2.0, 3.0]))

In [ ]:
from contextlib import contextmanager
from collections.abc import Iterator

@contextmanager
def timer(name: str) -> Iterator[None]:
    start = perf_counter()
    try:
        yield
    finally:
        elapsed_ms = (perf_counter() - start) * 1_000
        print(f"{name}: {elapsed_ms:.3f} ms")

with timer("构造数据"):
    values = [number * 2 for number in range(100_000)]

## 7. 异常处理

捕获具体异常，并在能够恢复、补充信息或转换抽象层时处理。使用 `raise ... from ...` 可以保留原始原因。

In [ ]:
def parse_world_size(raw: str) -> int:
    try:
        world_size = int(raw)
    except ValueError as error:
        raise ValueError(f"WORLD_SIZE 必须是整数，收到 {raw!r}") from error
    if world_size <= 0:
        raise ValueError("WORLD_SIZE 必须大于 0")
    return world_size

print(parse_world_size("8"))

## 8. 类型标注与不可变数据类

类型标注帮助编辑器、静态检查器和代码读者理解接口，但默认不会在运行时自动检查类型。

In [ ]:
from collections.abc import Sequence
from dataclasses import dataclass

@dataclass(frozen=True)
class ShardSpec:
    rank: int
    world_size: int

def shard(values: Sequence[int], spec: ShardSpec) -> Sequence[int]:
    if not 0 <= spec.rank < spec.world_size:
        raise ValueError("rank 必须位于 [0, world_size)")
    return values[spec.rank::spec.world_size]

numbers = list(range(12))
for rank in range(3):
    print(rank, shard(numbers, ShardSpec(rank, 3)))

# Python 并发与性能分析

先识别瓶颈，再选择工具：I/O 密集任务常用线程或异步；纯 Python 的 CPU 密集任务可考虑多进程；NumPy、PyTorch 和 CUDA 通常由原生代码或设备负责并行。

## 9. 线程与共享状态

常见 CPython 构建中的 GIL 会限制同一进程内多个线程同时执行 Python 字节码，但线程依然适用于阻塞 I/O，以及由原生扩展释放 GIL 的工作。共享可变状态需要正确同步。

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from time import sleep

def simulated_io(task_id: int) -> str:
    sleep(0.1)
    return f"任务 {task_id} 完成"

with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(simulated_io, range(4)))
print(results)

In [ ]:
from threading import Lock, Thread

counter = 0
lock = Lock()

def increment_many(times: int) -> None:
    global counter
    for _ in range(times):
        with lock:
            counter += 1

threads = [Thread(target=increment_many, args=(10_000,)) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print("counter =", counter)

## 10. 多进程

多进程拥有独立解释器，能够处理纯 Python 的 CPU 密集工作，但创建进程、序列化数据和进程间通信都有成本。Windows/Jupyter 的进程启动方式与普通脚本不同，工程代码应放入 `.py` 文件，并使用 `if __name__ == '__main__':` 保护入口。

```python
from concurrent.futures import ProcessPoolExecutor

def cpu_task(number: int) -> int:
    return sum(i * i for i in range(number))

if __name__ == '__main__':
    with ProcessPoolExecutor() as pool:
        results = list(pool.map(cpu_task, [1_000_000] * 4))
```

> GPU 提示：不要在父进程已经初始化 CUDA 后随意 `fork`；应遵循所用框架推荐的启动器与 start method。

## 11. `asyncio` 协作式并发

协程在 `await` 处主动让出控制权。事件循环中若直接执行阻塞 I/O 或长时间 CPU 计算，会阻塞其他协程。

In [ ]:
import asyncio

async def worker(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return name

async def run_workers() -> list[str]:
    return await asyncio.gather(
        worker("a", 0.2),
        worker("b", 0.1),
    )

# Jupyter 已经运行事件循环，因此直接使用 await。
print(await run_workers())

## 12. 基准测试

可靠测试需要预热和重复采样，不能只运行一次。GPU 操作通常异步提交；测量 GPU 时还需同步或使用框架提供的 Event。

In [ ]:
from collections.abc import Callable
from statistics import median
from time import perf_counter

def benchmark(function: Callable[[], object], warmup: int = 5, repeat: int = 20) -> float:
    for _ in range(warmup):
        function()
    samples: list[float] = []
    for _ in range(repeat):
        start = perf_counter()
        function()
        samples.append(perf_counter() - start)
    return median(samples)

data = list(range(10_000))
seconds = benchmark(lambda: sum(data))
print(f"中位耗时: {seconds * 1_000:.4f} ms")

## 13. CPU 与内存性能分析

推荐从端到端计时逐步缩小范围，再使用函数级、行级、内存和 GPU 专用分析工具。

In [ ]:
import cProfile
import io
import pstats

def workload() -> int:
    return sum(number * number for number in range(100_000))

profiler = cProfile.Profile()
profiler.enable()
workload()
profiler.disable()

stream = io.StringIO()
pstats.Stats(profiler, stream=stream).sort_stats("cumulative").print_stats(8)
print(stream.getvalue())

In [ ]:
import tracemalloc

tracemalloc.start()
temporary = [str(number) for number in range(20_000)]
snapshot = tracemalloc.take_snapshot()
for statistic in snapshot.statistics("lineno")[:5]:
    print(statistic)
tracemalloc.stop()

## 14. 性能优化顺序

1. 优先改善算法复杂度和数据结构。
2. 减少多余的 I/O、复制与序列化。
3. 用批处理或原生向量化算子替代细粒度 Python 循环。
4. 减少 Python 与 C++/GPU 边界的高频往返。
5. 用分析工具确认热点后，再考虑 C++、CUDA 或 Triton。

核心原则：先测量，再优化。对 AI Infra 而言，批量处理往往比微调 Python 语法更有效。

## 练习

1. 修改 `Batch`，实现 `__getitem__`，使其支持索引访问。
2. 为 `make_batches` 添加丢弃最后一个不完整批次的选项。
3. 比较串行执行与线程池执行模拟 I/O 的耗时。
4. 修改 `benchmark`，同时返回最小值、中位数和最大值。
5. 使用 `cProfile` 找出你自己一段程序中累计耗时最高的函数。